In [ ]:
import pandas as pd
import plotly.express as px
import plotly
from pathlib import Path

# ── constants ────────────────────────────────────────────────────────────────

LANG2S = {
    "svo": {"eng": "verb", "zho": "verb", "rus": "verb", "tur": "object", "jap": "object"},
    "modal-verbs": {"eng": "modal", "zho": "modal", "tur": "verb", "ger": "modal"},
    "noun-adj": {
        "eng": "adj", "ger": "adj", "rus": "adj", "zho": "adj", "ned": "adj",
        "fre": "noun", "ita": "noun", "vie": "noun",
    },
    "noun-adj-complete": {
        "eng": "adj", "ger": "adj", "rus": "adj", "zho": "adj", "ned": "adj",
        "fre": "noun", "ita": "noun", "vie": "noun",
    },
}
EXPERIMENT2BASE_LANGS = {
    "noun-adj-complete": ["fre", "ita", "vie"],
    "svo": LANG2S["svo"].keys(),
    "modal-verbs": LANG2S["modal-verbs"].keys(),
}
EXPERIMENT2PLANT_LANGS = {
    "noun-adj-complete": ["eng", "ger", "ned", "rus", "zho"],
    "svo": LANG2S["svo"].keys(),
    "modal-verbs": LANG2S["modal-verbs"].keys(),
}

MODEL2LAYER    = {"mGPT": 11, "aya-expanse-8b": 15, "Meta-Llama-3-8B": 14}
MODEL2NUMHEADS = {"mGPT": 16, "aya-expanse-8b": 32, "Meta-Llama-3-8B": 32}

# ── helpers ──────────────────────────────────────────────────────────────────

def get_valid_lang_pairs(experiment, base_langs, plant_langs, src_lang):
    """Return (base, plant) pairs where both are in the experiment and have different syntax roles."""
    lang2s = LANG2S[experiment]
    return [
        (b, p)
        for b in base_langs
        for p in plant_langs
        if b != p
        and b in lang2s
        and p in lang2s
        and b != src_lang
        and p != src_lang
        and lang2s[b] != lang2s[p]
    ]


def load_df(path, file_format, experiment, model, src_lang, tgt_lang_base, tgt_lang_plant, layer):
    """Load one CSV, keep only the target layer, and add derived columns."""
    file = file_format.format(
        EXPERIMENT=experiment, MODEL=model, SRC_LANG=src_lang,
        tgt_lang_base=tgt_lang_base, tgt_lang_plant=tgt_lang_plant,
    )
    df = pd.read_csv(path / file)
    df = df[df["layer"] == layer].copy()

    df["part_of_speech+lexical_component"] = (
        df["part_of_speech"] + "+" + df["lexical_component"]
    )
    # Average probability across all heads for the same token position
    df["avg_prob_per_layer"] = df.groupby(
        ["token_type", "layer", "sentence_id"]
    )["prob"].transform("mean")
    df["prob_ratio"] = df["prob"] / df["avg_prob_per_layer"]
    return df


def get_heatmap(df, layer, head):
    """Pivot a single head's data into a (language × PoS+concept) heatmap."""
    head_df = (
        df[(df["layer"] == layer) & (df["head_id"] == head)]
        .groupby(["language", "part_of_speech+lexical_component"], as_index=False)
        .mean(numeric_only=True)
    )
    return head_df.pivot(
        index="language",
        columns="part_of_speech+lexical_component",
        values="prob_ratio",
    )

def normalize_heatmap(hm, tgt_lang_base, tgt_lang_plant, experiment):
    """
    Replace concrete language/pos labels with abstract base/plant labels,
    so heatmaps from different pairs can be meaningfully averaged.
    """
    base_s  = LANG2S[experiment][tgt_lang_base]
    plant_s = LANG2S[experiment][tgt_lang_plant]

    hm = hm.rename(index={tgt_lang_base: "base L", tgt_lang_plant: "plant L"})
    hm = hm.rename(columns={
        f"{base_s}+base":   "base S+base C",
        f"{base_s}+plant":  "base S+plant C",
        f"{plant_s}+base":  "plant S+base C",
        f"{plant_s}+plant": "plant S+plant C",
    })
    # Keep only the four canonical columns (drop anything unexpected)
    canonical = ["base S+base C", "base S+plant C", "plant S+base C", "plant S+plant C"]
    return hm.reindex(index=["base L", "plant L"], columns=canonical)


def aggregate_heatmaps(path, file_format, experiment, model, src_lang, lang_pairs, layer, num_heads):
    heatmaps = {}
    for tgt_lang_base, tgt_lang_plant in lang_pairs:
        df = load_df(path, file_format, experiment, model, src_lang,
                     tgt_lang_base, tgt_lang_plant, layer)
        for head in range(num_heads):
            hm = get_heatmap(df, layer=layer, head=head)
            hm = normalize_heatmap(hm, tgt_lang_base, tgt_lang_plant, experiment)
            heatmaps[head] = hm if head not in heatmaps else heatmaps[head].add(hm, fill_value=0)

    n = len(lang_pairs)
    return {head: hm / n for head, hm in heatmaps.items()}


def is_outlier(heatmap, plant_s):
    """
    True when, pooled across all languages, plant-syntax + plant-concept > 1.
    'Regardless of L' means we check every row (language) with .any().
    """
    return (heatmap["plant S+base C"] > 1.0).all()


def plot_head(heatmap, experiment, layer, head, filename_stem):
    col_order = ["base S+base C", "base S+plant C", "plant S+base C", "plant S+plant C"]
    col_order = [c for c in col_order if c in heatmap.columns]  # keep only present columns

    fig = px.imshow(
        heatmap[col_order],
        text_auto=True,
        labels={"x": "PoS + Concept", "y": "Language", "color": "Probability Ratio"},
        title=f"[{experiment}] Probability Ratio — Head {layer}.{head}",
        color_continuous_scale="plasma",
    )
    out_path = f"{filename_stem}_layer{layer}_head{head}.html"
    plotly.offline.plot(fig, filename=out_path, auto_open=False)
    return fig
    # print(f"  Saved: {out_path}")


# ── main loop ────────────────────────────────────────────────────────────────

PATH        = Path("output/intervention/probs")          # adjust as needed
SRC_LANG    = "eng"
EXPERIMENTS = ["noun-adj-complete", "svo", "modal-verbs"]
MODELS      = ["mGPT", "aya-expanse-8b", "Meta-Llama-3-8B"]
file_format = "attn-heads_{EXPERIMENT}_{MODEL}_{SRC_LANG}-{tgt_lang_base}_{SRC_LANG}-{tgt_lang_plant}.csv"  # adjust


# BASE_LANGS  = ["fre", "ita", "vie"]
# PLANT_LANGS = ["rus", "zho", "ger", "ned"]

for experiment in EXPERIMENTS:
    BASE_LANGS = EXPERIMENT2BASE_LANGS[experiment]
    PLANT_LANGS = EXPERIMENT2PLANT_LANGS[experiment]
    for model in MODELS:
        layer     = MODEL2LAYER[model]
        num_heads = MODEL2NUMHEADS[model]

        lang_pairs = get_valid_lang_pairs(experiment, BASE_LANGS, PLANT_LANGS, SRC_LANG)
        if not lang_pairs:
            continue
            
        print(f"\n{'='*60}")
        print(f"Experiment: {experiment} | Model: {model} | Layer: {layer}")
        print(f"Valid lang pairs ({len(lang_pairs)}): {lang_pairs}")
            
        heatmaps = aggregate_heatmaps(
            PATH, file_format, experiment, model, SRC_LANG,
            lang_pairs, layer, num_heads,
        )

        # print(heatmaps[2])

        filename_stem = f"output/intervention/figures/attn-heads_{experiment}_{model}_src-{SRC_LANG}"

        outlier_heads = []
        for head, hm in heatmaps.items():
            # plant_s is determined by the experiment, not by any single language pair
            # We use the first plant language that is in this experiment as the reference
            plant_s = next(
                LANG2S[experiment][p] for p in PLANT_LANGS if p in LANG2S[experiment]
            )
            base_s = next(
                LANG2S[experiment][b] for b in PLANT_LANGS if b in LANG2S[experiment]
            )
            if is_outlier(hm, plant_s):
                outlier_heads.append(head)
                # print(f"  Outlier head {layer}.{head}")
                fig = plot_head(hm, experiment, layer, head, filename_stem)
                fig.show()

        if not outlier_heads:
            print("  No outlier heads found.")
        else:
            print(f"  Outlier heads: {outlier_heads}")